# Documents Corpus Review (read-only)

Human-validation surface for the documents eval corpus. Each row is **routed by
the real model and executed for real** against a fresh sandbox copy of the
fixtures, then graded with the skill's `honest()` `success_criteria` logic. A
scannable green/red table shows, per row: expected vs. actual tool, outcome, the
criteria score, and the matched/failed detail.

Why not `baseline_authoring.ipynb`? That notebook surfaces ffmpeg-style baseline
commands / multi-output chains via `needs_review()`; documents rows are
`success_criteria`-graded with `baseline: null`, so they never appear there. And
the eval suite's `honest` verifier grades a *dry-run* dict, so `output_exists`
cannot pass for destructive tools. This reviewer executes for real instead, so
the pass/fail you see is the truth. **Nothing is written to the repo or corpus.**

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), Path.cwd())
HELPERS = ROOT / "src" / "skills" / "documents" / "notebooks"
for _p in (str(ROOT), str(HELPERS)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("Root:", ROOT)

In [ ]:
# qwen3-4b (mirrors eval_backends.yaml / models.yaml). Swap path for another GGUF.
MODEL_CFG = {
    "path": "models/Qwen3-4B-Q4_K_M.gguf",
    "n_gpu_layers": 99,
    "n_ctx": 8192,
    "n_threads": 8,
    "max_tokens": 2048,
    "json_mode": False,
    "thinking_enabled": False,
}

In [ ]:
from helpers.documents_corpus_review import render_review, review_corpus

# First run executes the model over every row (~0.3-2s each) and caches results
# to notebooks/artifacts/documents_review.json. Re-runs - even after a kernel
# restart - load the cache instantly. Set refresh=True to force a fresh run
# (do this after changing the corpus or the model).
rows = review_corpus(ROOT, MODEL_CFG, limit=None, refresh=True)
render_review(rows)

In [ ]:
# Just the rows that need a look (red in the table above).
for r in rows:
    tool_ok = r["actual_tool"] == r["expected_tool"] or (
        r["expected_outcome"] in ("clarify", "reject")
        and r["actual_outcome"] == r["expected_outcome"]
    )
    crit_ok = r.get("score") is None or r["score"] >= 1.0
    if not (r["outcome_ok"] and tool_ok and crit_ok):
        print(
            f"{r['id']:16} {r['utterance'][:45]:45} exp={r['expected_tool'] or r['expected_outcome']:16} "
            f"got={r['actual_tool']}/{r['actual_outcome']}  {r['failed']}"
        )